<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Finding How The Data Is Distributed**


Estimated time needed: **30** minutes


In this lab, you will work with a cleaned dataset to perform Exploratory Data Analysis (EDA). You will examine the structure of the data, visualize key variables, and analyze trends related to developer experience, tools, job satisfaction, and other important aspects.


## Objectives


In this lab you will perform the following:


- Understand the structure of the dataset.

- Perform summary statistics and data visualization.

- Identify trends in developer experience, tools, job satisfaction, and other key variables.


### Install the required libraries


In [ ]:
!pip install pandas
!pip install matplotlib
!pip install seaborn


### Step 1: Import Libraries and Load Data


- Import the `pandas`, `matplotlib.pyplot`, and `seaborn` libraries.


- You will begin with loading the dataset. You can use the pyfetch method if working on JupyterLite. Otherwise, you can use pandas' read_csv() function directly on their local machines or cloud environments.


In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the Stack Overflow survey dataset
data_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv'
df = pd.read_csv(data_url)

# Display the first few rows of the dataset
df.head()


### Step 2: Examine the Structure of the Data


- Display the column names, data types, and summary information to understand the data structure.

- Objective: Gain insights into the dataset's shape and available variables.


In [ ]:
## Write your code here
# Column names, data types, and summary info
print(df.columns)
print(df.dtypes)
print(df.info())

# Basic statistics for numeric columns
print(df.describe())

### Step 3: Handle Missing Data


- Identify missing values in the dataset.

- Impute or remove missing values as necessary to ensure data completeness.



In [ ]:
## Write your code here
# Count missing values per column
missing_values = df.isnull().sum().sort_values(ascending=False)
print(missing_values.head(10))

# Fill missing values in critical columns with mode
for col in ['Employment', 'JobSat', 'RemoteWork']:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Drop rows where YearsCodePro is missing (if needed)
df = df.dropna(subset=['YearsCodePro'])

# Verify missing values handled
print(df[['Employment', 'JobSat', 'RemoteWork', 'YearsCodePro']].isnull().sum())

### Step 4: Analyze Key Columns


- Examine key columns such as `Employment`, `JobSat` (Job Satisfaction), and `YearsCodePro` (Professional Coding Experience).

- **Instruction**: Calculate the value counts for each column to understand the distribution of responses.



In [ ]:
## Write your code here
# Employment distribution
print(df['Employment'].value_counts())

# Job satisfaction distribution
print(df['JobSat'].value_counts())

# Years of professional coding experience
print(df['YearsCodePro'].value_counts().head(10))  # Top 10 most frequent

### Step 5: Visualize Job Satisfaction (Focus on JobSat)


- Create a pie chart or KDE plot to visualize the distribution of `JobSat`.

- Provide an interpretation of the plot, highlighting key trends in job satisfaction.


In [ ]:
## Write your code here
# Pie chart for JobSat
job_counts = df['JobSat'].value_counts()
plt.figure(figsize=(8,8))
plt.pie(job_counts, labels=job_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title('Job Satisfaction Distribution')
plt.show()

### Step 6: Programming Languages Analysis


- Compare the frequency of programming languages in `LanguageHaveWorkedWith` and `LanguageWantToWorkWith`.
  
- Visualize the overlap or differences using a Venn diagram or a grouped bar chart.


In [ ]:
## Write your code here
# Split multiple languages and explode
df_lang_worked = df[['LanguageHaveWorkedWith']].dropna()
df_lang_worked['LanguageHaveWorkedWith'] = df_lang_worked['LanguageHaveWorkedWith'].str.split(';')
df_lang_worked = df_lang_worked.explode('LanguageHaveWorkedWith')

df_lang_want = df[['LanguageWantToWorkWith']].dropna()
df_lang_want['LanguageWantToWorkWith'] = df_lang_want['LanguageWantToWorkWith'].str.split(';')
df_lang_want = df_lang_want.explode('LanguageWantToWorkWith')

# Top 10 languages worked with
top_worked = df_lang_worked['LanguageHaveWorkedWith'].value_counts().head(10)
top_want = df_lang_want['LanguageWantToWorkWith'].value_counts().head(10)

# Grouped bar chart
top_langs = pd.DataFrame({'Worked': top_worked, 'WantToWork': top_want})
top_langs.plot(kind='bar', figsize=(12,6))
plt.title('Top Programming Languages: Worked With vs Want to Work With')
plt.ylabel('Number of Respondents')
plt.show()

### Step 7: Analyze Remote Work Trends


- Visualize the distribution of RemoteWork by region using a grouped bar chart or heatmap.


In [ ]:
## Write your code here
# Cross-tab by Country
remote_by_country = pd.crosstab(df['Country'], df['RemoteWork'])
remote_by_country_top = remote_by_country.sort_values(by='Fully remote', ascending=False).head(10)

# Heatmap
plt.figure(figsize=(12,6))
sns.heatmap(remote_by_country_top, annot=True, fmt='d', cmap='coolwarm')
plt.title('Remote Work Distribution by Top 10 Countries')
plt.ylabel('Country')
plt.xlabel('Remote Work Preference')
plt.show()

### Step 8: Correlation between Job Satisfaction and Experience


- Analyze the correlation between overall job satisfaction (`JobSat`) and `YearsCodePro`.
  
- Calculate the Pearson or Spearman correlation coefficient.


In [ ]:
## Write your code here
# Convert JobSat to numeric if available
job_sat_map = {
    'Very dissatisfied': 1,
    'Slightly dissatisfied': 2,
    'Neither satisfied nor dissatisfied': 3,
    'Slightly satisfied': 4,
    'Very satisfied': 5
}
df['JobSatPoints'] = df['JobSat'].map(job_sat_map)

# Convert YearsCodePro to numeric
df['YearsCodePro_num'] = pd.to_numeric(df['YearsCodePro'], errors='coerce')

# Calculate Pearson correlation
corr = df[['YearsCodePro_num', 'JobSatPoints']].corr(method='pearson')
print(f"Pearson correlation between experience and job satisfaction: \n{corr}")

# Scatter plot
plt.figure(figsize=(8,5))
sns.scatterplot(x='YearsCodePro_num', y='JobSatPoints', data=df, alpha=0.5)
plt.title('Experience vs Job Satisfaction')
plt.xlabel('Years of Professional Coding')
plt.ylabel('Job Satisfaction Points')
plt.show()

### Step 9: Cross-tabulation Analysis (Employment vs. Education Level)


- Analyze the relationship between employment status (`Employment`) and education level (`EdLevel`).

- **Instruction**: Create a cross-tabulation using `pd.crosstab()` and visualize it with a stacked bar plot if possible.


In [ ]:
## Write your code here
# Cross-tab
edu_emp_crosstab = pd.crosstab(df['EdLevel'], df['Employment'])
print(edu_emp_crosstab)

# Stacked bar plot
edu_emp_crosstab.plot(kind='bar', stacked=True, figsize=(12,6), colormap='Paired')
plt.title('Employment Type by Education Level')
plt.ylabel('Number of Respondents')
plt.xlabel('Education Level')
plt.show()

### Step 10: Export Cleaned Data


- Save the cleaned dataset to a new CSV file for further use or sharing.


In [ ]:
## Write your code here
df.to_csv('StackOverflow_EDA_Cleaned.csv', index=False)
print("Cleaned dataset saved successfully.")

### Summary:


In this lab, you practiced key skills in exploratory data analysis, including:


- Examining the structure and content of the Stack Overflow survey dataset to understand its variables and data types.

- Identifying and addressing missing data to ensure the dataset's quality and completeness.

- Summarizing and visualizing key variables such as job satisfaction, programming languages, and remote work trends.

- Analyzing relationships in the data using techniques like:
    - Comparing programming languages respondents have worked with versus those they want to work with.
      
    - Exploring remote work preferences by region.

- Investigating correlations between professional coding experience and job satisfaction.

- Performing cross-tabulations to analyze relationships between employment status and education levels.


## Authors:
Ayushi Jain


### Other Contributors:
Rav Ahuja
Lakshmi Holla
Malika


Copyright © IBM Corporation. All rights reserved.
